# Linear Regression for Appliance Energy Use

This notebook uses ordinary least squares to estimate `Appliances`, the energy used by household appliances in watt-hours (Wh) during each 10-minute interval. It uses the repository's `energydata.csv` from the UCI Appliances Energy Prediction dataset.

Put this notebook next to `energydata.csv` in the repository root. Install the dependencies with `python -m pip install -r requirements.txt`, select that Python environment as the notebook kernel, and run all cells.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

## Load and inspect the data

Check for missing values and duplicate rows before fitting the model.

In [ ]:
data_path = Path("energydata.csv")
if not data_path.is_file():
    raise FileNotFoundError("Open the notebook from the repository root next to energydata.csv.")

data = pd.read_csv(data_path, parse_dates=["date"])
print(f"Rows: {len(data):,} | Columns: {data.shape[1]}")
print(f"Missing values: {data.isna().sum().sum():,} | Duplicate rows: {data.duplicated().sum():,}")
print(f"Date range: {data['date'].min()} to {data['date'].max()}")
display(data[["date", "Appliances", "lights", "T1", "RH_1", "T_out"]].head())

## Prepare features and split the data

Use indoor sensor readings, weather measurements, lighting use, and calendar features. The calendar fields follow the group's KNN and MLP preprocessing. The dataset's observations all come from 2016, so the constant year adds no information. The dataset supplies `rv1` and `rv2` as random variables, so both are excluded.

The shuffled 70/30 split with seed 42 uses the same test rows as the KNN and MLP notebooks. Since rows close in time can appear on both sides of the split, these scores measure performance on held-out intervals from the same observation period. Forecasting later periods would require a chronological holdout and inputs available at prediction time.

In [ ]:
timestamp = data["date"]
X = data.drop(columns=["date", "Appliances", "rv1", "rv2"]).assign(
    month=timestamp.dt.month,
    day=timestamp.dt.day,
    hour=timestamp.dt.hour,
    minute=timestamp.dt.minute,
)
y = data["Appliances"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)
print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Predictors: {X.shape[1]}")

## Train and evaluate the model

Ordinary least squares fits the intercept and coefficients that minimize squared prediction error on the training set. Compare it with a baseline that predicts the training-set mean for every test row.

MAE gives the average absolute error in Wh. MSE is measured in Wh², and its square root (RMSE) is in Wh. RMSE gives greater weight to large errors. R² measures how much of the variation in test usage the model explains.

In [ ]:
linear_model = LinearRegression().fit(X_train, y_train)
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)

predicted = linear_model.predict(X_test)
baseline_predicted = baseline.predict(X_test)

def evaluate(actual, estimates):
    return {
        "MAE (Wh)": mean_absolute_error(actual, estimates),
        "MSE (Wh²)": mean_squared_error(actual, estimates),
        "RMSE (Wh)": root_mean_squared_error(actual, estimates),
        "R²": r2_score(actual, estimates),
    }

metrics = pd.DataFrame.from_dict(
    {
        "Training-mean baseline": evaluate(y_test, baseline_predicted),
        "Linear regression": evaluate(y_test, predicted),
    },
    orient="index",
)
display(metrics.round(3))

baseline_rmse = metrics.loc["Training-mean baseline", "RMSE (Wh)"]
model_rmse = metrics.loc["Linear regression", "RMSE (Wh)"]
rmse_reduction = 100 * (baseline_rmse - model_rmse) / baseline_rmse
print(f"Test RMSE reduction relative to baseline: {rmse_reduction:.1f}%")

## Inspect test predictions

The diagonal in the first plot marks perfect predictions. In the second plot, a negative residual (predicted minus actual) means the model underpredicted energy use.

In [ ]:
residuals = predicted - y_test.to_numpy()
low = min(y_test.min(), predicted.min())
high = max(y_test.max(), predicted.max())

fig, (actual_ax, residual_ax) = plt.subplots(1, 2, figsize=(11, 4.3), layout="constrained")
actual_ax.scatter(y_test, predicted, s=12, alpha=0.25, color="#176A7D")
actual_ax.plot([low, high], [low, high], color="#4A4A4A", linestyle="--")
actual_ax.set(
    title="Actual vs. predicted usage",
    xlabel="Actual appliance energy (Wh)",
    ylabel="Predicted appliance energy (Wh)",
    xlim=(low, high),
    ylim=(low, high),
)
actual_ax.set_aspect("equal", adjustable="box")

residual_ax.scatter(y_test, residuals, s=12, alpha=0.25, color="#176A7D")
residual_ax.axhline(0, color="#4A4A4A", linestyle="--")
residual_ax.set(
    title="Residuals by actual usage",
    xlabel="Actual appliance energy (Wh)",
    ylabel="Prediction minus actual (Wh)",
)
plt.show()

In [ ]:
high_use_threshold = y_train.quantile(0.90)
high_use = y_test.to_numpy() >= high_use_threshold
print(f"High-use threshold (training 90th percentile): {high_use_threshold:.0f} Wh")
print(f"Mean signed error for {high_use.sum():,} high-use intervals: {residuals[high_use].mean():.1f} Wh")
print(f"Mean signed error for other intervals: {residuals[~high_use].mean():.1f} Wh")

## Interpret the coefficients

A coefficient estimates the change in predicted appliance energy (Wh) for a one-unit increase in that feature while holding the other inputs fixed. Feature units differ, so the magnitude of a coefficient depends on the units. Correlated readings can make individual coefficients sensitive to which features are included.

In [ ]:
coefficients = pd.Series(linear_model.coef_, index=X.columns, name="Coefficient (Wh per unit)")
display(coefficients.round(3).to_frame())

## Conclusion

Linear regression improves on the training-mean baseline, but it explains only a modest share of the variation in test usage. The residuals show that it tends to underestimate high-use intervals. Use its test scores as a reference when comparing the other group models on the same split.